# IOI + the Hydra effect in GPT-2 small

**Question.** GPT-2 small solves Indirect Object Identification (IOI) with a known circuit ([Wang et al. 2022](https://arxiv.org/abs/2211.00593)).
When we ablate the heads that *directly* write the answer (the name movers), how much of the loss does the rest of the network make up for
([McGrath et al. 2023, the Hydra effect](https://arxiv.org/abs/2307.15771)), which heads do the repairing, and how much do the answers depend on
the choice of ablation (a systematic uncertainty)?

**Plan.**
1. Dataset and baseline, with bootstrap CIs over prompts
2. Residual-stream activation patching (layer × position)
3. Per-head activation patching: recover the circuit
4. Direct logit attribution (DLA) and attention patterns: name movers vs negative name movers
5. **Hydra:** naive direct-effect prediction vs actual drop, and who compensates
6. **Additivity:** single-head ablation drops vs joint ablation drop
7. **Systematics:** ablation method, template, dataset seed
8. Write-up

Helpers live in `ioi_hydra.py`. Everything is computed per prompt so it can be bootstrapped.
The full notebook runs in about 5 minutes on CPU on an M-series Mac; the head-patching cell is the slow one.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, torch, matplotlib.pyplot as plt
from transformer_lens.model_bridge import TransformerBridge
from ioi_hydra import *
from pathlib import Path

FIGDIR = Path("../docs/figures")  # figures used by docs/results.md
FIGDIR.mkdir(parents=True, exist_ok=True)
savefig = lambda name: plt.savefig(FIGDIR / f"{name}.png", dpi=150, bbox_inches="tight")

DEVICE = "cpu"  # "mps" also works on Apple silicon; CPU is plenty for GPT-2 small
model = TransformerBridge.boot_transformers("gpt2", device=DEVICE)
model.enable_compatibility_mode()  # fold LayerNorm, etc. -> numerics match legacy HookedTransformer / the papers
L, H = model.cfg.n_layers, model.cfg.n_heads

## 1. Dataset and baseline

8 templates × 16 prompts, balanced between ABBA and BABA name orders. Each clean prompt has a matched **ABC** corrupt prompt
(same template, three fresh names), which carries no information about which name is the IO.
Metric: logit difference LD = logit(IO) − logit(S) at the final token. The IOI paper reports about 3.5 on its own dataset.

In [ ]:
ds = make_ioi_dataset(model, n_per_template=16, seed=0)
for i in range(4):
    print(f"{ds.order[i]}  {ds.prompts[i]!r:80}  ->  {model.to_string(ds.io[i])!r}")
    print(f"      {ds.corrupt_prompts[i]!r}")

clean_logits, clean_cache = model.run_with_cache(ds.clean_tokens)
corrupt_logits, corrupt_cache = model.run_with_cache(ds.corrupt_tokens)
clean_ld, corrupt_ld = logit_diff(clean_logits, ds), logit_diff(corrupt_logits, ds)
CLEAN, CORRUPT = clean_ld.mean().item(), corrupt_ld.mean().item()

print(f"\nclean LD   {fmt_ci(bootstrap_ci(clean_ld))}   (fraction LD > 0: {(clean_ld > 0).float().mean():.3f})")
print(f"corrupt LD {fmt_ci(bootstrap_ci(corrupt_ld))}")

In [ ]:
# Template-to-template spread is a systematic: is it larger than the statistical error on each template?
for t, tmpl in enumerate(TEMPLATES):
    print(f"{t}  LD {fmt_ci(bootstrap_ci(clean_ld[ds.template_id == t]))}   {tmpl}")

## 2. Residual-stream patching (layer × position)

Denoising: run the **corrupt** prompt, patch in the **clean** `resid_pre` at one (layer, position), and measure normalized LD
(0 = corrupt, 1 = clean). Uses template 0 only, so positions line up, and splits ABBA from BABA. The IO and S1 slots swap
between the two orders, so averaging them together would cancel the name-position effects. Normalization uses each
subset's *own* clean and corrupt LD. The colour scale is not clipped: values above 1 mean the patch does *better* than the
clean run.

*Expect:* early on the information sits at the names (IO, S1, S2). Around layers 8–10 it moves to the final token.

In [ ]:
ds0 = ds.subset(ds.template_id == 0)
resid_patch, labels = {}, {}
for order in ["ABBA", "BABA"]:
    dso = ds0.subset(torch.tensor([o == order for o in ds0.order]))
    c_logits, c_cache = model.run_with_cache(dso.clean_tokens)
    k_logits, _ = model.run_with_cache(dso.corrupt_tokens)
    C0, K0 = logit_diff(c_logits, dso).mean().item(), logit_diff(k_logits, dso).mean().item()
    print(f"{order}: clean LD {C0:.2f}, corrupt LD {K0:.2f}")
    resid_patch[order] = patch_resid_by_pos(model, dso, c_cache, C0, K0)
    # Name slots are labelled by role, since the names differ across prompts.
    roles = {int(dso.pos[k][0]): k for k in ["IO", "S1", "S2"]}
    labels[order] = [f"{i}:{roles.get(i, t)}" for i, t in enumerate(model.to_str_tokens(dso.clean_tokens[0]))]

vmax = max(r.abs().max().item() for r in resid_patch.values())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, order in zip(axes, resid_patch):
    heatmap(resid_patch[order], f"Residual patching, {order} (normalized LD)", xlabel="position",
            xticklabels=labels[order], ax=ax, vmax=vmax)
plt.tight_layout()
for order, rp in resid_patch.items():
    dso = ds0.subset(torch.tensor([o == order for o in ds0.order]))
    print(f"\n{order}            " + " ".join(f"L{l:<5}" for l in range(L)))
    for role in ["IO", "S1", "S2", "end"]:
        p = int(dso.pos[role][0])
        print(f"  {role:3} (pos {p:2})  " + " ".join(f"{v:+.2f} " for v in rp[:, p].tolist()))
    others = [p for p in range(rp.shape[1]) if p not in {int(dso.pos[k][0]) for k in dso.pos}]
    print(f"  max |value| at all other positions: {rp[:, others].abs().max():.2f}")
savefig("resid_patching")

## 3. Per-head patching

Patch each head's clean output (`hook_z`, all positions) into the corrupt run. This takes about 2 minutes on CPU.

*Compare with the paper:* name movers 9.9, 9.6, 10.0; S-inhibition 7.3, 7.9, 8.6, 8.10; induction / duplicate-token heads in
layers 0–6; backup name movers 10.10, 10.6, 10.2, 10.1, 11.2, 9.7, 9.0, 11.9; negative name movers 10.7, 11.10.

In [ ]:
head_patch = patch_heads(model, ds, clean_cache, CLEAN, CORRUPT)   # [N, L, H]
heatmap(head_patch.mean(0), "Head patching: clean hook_z -> corrupt run (normalized LD)", annotate_top=10)
for l, h, v in top_heads(head_patch.mean(0).abs(), 12):
    print(f"{l}.{h:<3} {fmt_ci(bootstrap_ci(head_patch[:, l, h]), 3)}")
savefig("head_patching")

## 4. Direct logit attribution

Each head's direct contribution to LD at the final token (output → final LN scale → unembed). Name movers should be strongly positive
and attend END → IO. Negative name movers should be negative and also attend to IO, pushing *down* the name they copy.

In [ ]:
de_clean = head_direct_effects(model, clean_cache, ds)    # [N, L, H]
heatmap(de_clean.mean(0), "Direct effect on LD per head (clean run)", annotate_top=10)

print("head   DE                     attn END->IO   attn END->S1")
for l, h, _ in top_heads(de_clean.mean(0).abs(), 8):
    print(f"{l}.{h:<4} {fmt_ci(bootstrap_ci(de_clean[:, l, h])):22} "
          f"{attention_to(clean_cache, ds, l, h, 'IO').mean():.2f}           "
          f"{attention_to(clean_cache, ds, l, h, 'S1').mean():.2f}")
savefig("direct_effects")

**Do these heads copy?** For each head, take the residual stream at the IO position as the head reads it, pass it
through the head's OV circuit, and unembed. The copy score is the fraction of prompts where the IO token is among the
5 highest logits over the whole vocabulary; the negative copy score, among the 5 lowest (Wang et al.). This measures
only what a head *would* write if it attended to the name. A name mover needs both copying and attention to IO.

In [ ]:
copy, neg_copy = copy_scores(model, clean_cache, ds)

ROLES = {"name mover": [(9, 9), (9, 6), (10, 0)],
         "backup": [(10, 10), (10, 6), (10, 2), (10, 1), (11, 2)],
         "brake": [(10, 7), (11, 10)]}
print("head   role         copy   neg copy   attn END->IO   DE")
for role, heads in ROLES.items():
    for l, h in heads:
        print(f"{l}.{h:<4} {role:11}  {copy[l, h]:.2f}   {neg_copy[l, h]:.2f}       "
              f"{attention_to(clean_cache, ds, l, h, 'IO').mean():.2f}           {de_clean[:, l, h].mean():+.2f}")
print("\nother heads with copy score > 0.9:",
      [f"{l}.{h}" for l, h, v in top_heads(copy, 20) if v > 0.9 and (l, h) not in sum(ROLES.values(), [])])

**Where do they look, and how does that change when the name movers are removed?** Attention from the final token
to IO, S1 and S2, in the clean run and with the name movers mean-ablated (§5). The ablated heads are hatched because
their outputs are replaced, so they no longer write anything. Ablation replaces a head's *output*, so 9.9's and
9.6's attention is unchanged; 10.0's does change, because it reads layer 9's output.

In [ ]:
ablator = Ablator(model, ds, corrupt_cache)
NAME_MOVERS = [(9, 9), (9, 6), (10, 0)]
_, nm_ablated_cache = ablator.run(NAME_MOVERS, "mean", with_cache=True)

heads = sum(ROLES.values(), [])
profiles = {"clean": attention_profile(clean_cache, ds, heads),
            "name movers ablated": attention_profile(nm_ablated_cache, ds, heads)}
COLORS = {"IO": "#2a78d6", "S1": "#eb6834", "S2": "#1baf7a"}  # categorical slots 1-3 (dataviz default palette)

# Leave a gap between role groups.
xs, x = [], 0.0
for role, hs in ROLES.items():
    xs += [x + i for i in range(len(hs))]
    x += len(hs) + 0.8
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
w = 0.26
for ax, (title, prof) in zip(axes, profiles.items()):
    for j, key in enumerate(["IO", "S1", "S2"]):
        vals = [prof[hd][key] for hd in heads]
        bars = ax.bar([xi + (j - 1) * (w + 0.02) for xi in xs], vals, w, color=COLORS[key], label=key, zorder=2)
        if title != "clean":
            for bar, hd in zip(bars, heads):
                if hd in NAME_MOVERS:
                    bar.set_alpha(0.35)
                    bar.set_hatch("///")
    ax.set_xticks(xs, [f"{l}.{h}" for l, h in heads])
    start = 0
    for role, hs in ROLES.items():
        mid = (xs[start] + xs[start + len(hs) - 1]) / 2
        ax.text(mid, -0.17, role, ha="center", va="top", transform=ax.get_xaxis_transform(), color="#52514e")
        start += len(hs)
    ax.set(title=title, ylim=(0, 1))
    ax.grid(axis="y", color="#e5e4df", zorder=0)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("attention from final token")
axes[0].legend(title="attends to", frameon=False, loc="upper right")
plt.tight_layout()
savefig("attention")

print("head    IO clean -> ablated   S1 clean -> ablated")
for hd in heads:
    c, a = profiles["clean"][hd], profiles["name movers ablated"][hd]
    print(f"{hd[0]}.{hd[1]:<4}  {c['IO']:.2f} -> {a['IO']:.2f}          {c['S1']:.2f} -> {a['S1']:.2f}")

**§3 vs §4.** In the ABC context nothing downstream can react to a patch, so per-head patching should equal
DE / (clean − corrupt):
$LD_{\text{patched}} - LD_{\text{corrupt}} = (DE_h^{\text{clean}} - DE_h^{\text{corrupt}}) + \Delta_{\text{downstream}} \approx DE_h^{\text{clean}}$.

In [ ]:
de_norm = de_clean.mean(0) / (CLEAN - CORRUPT)
print(f"correlation over all {L * H} heads: {np.corrcoef(de_norm.flatten(), head_patch.mean(0).flatten())[0, 1]:.3f}")
# Per prompt, measure each patch relative to that prompt's own corrupt LD, so that the prompt's baseline offset
# is not counted 144 times in the sum.
per_prompt = (head_patch - normalized(corrupt_ld, CLEAN, CORRUPT)[:, None, None]).sum((1, 2))
print(f"sum of patching values over all heads (signed): {fmt_ci(bootstrap_ci(per_prompt))}")
print("\nhead   patching   DE / gap")
for l, h, _ in top_heads(head_patch.mean(0).abs(), 8):
    print(f"{l}.{h:<4}  {head_patch[:, l, h].mean():+.3f}    {de_norm[l, h]:+.3f}")

# The same head, removed from the clean context instead (necessity rather than sufficiency):
single = clean_ld - logit_diff(Ablator(model, ds, corrupt_cache).run([(9, 9)], "mean")[0], ds)
print(f"\nmean-ablating 9.9 alone drops LD by {fmt_ci(bootstrap_ci(single))}  (its DE: {de_clean[:, 9, 9].mean():.2f})")

## 5. The Hydra effect

Ablate the three name movers. If the network were a sum of independent parts, LD would drop by the **direct effect** those heads lose
(the naive prediction). The **total effect** is what actually happens. Their gap is the **repair**:

$$\text{repair} = \underbrace{\textstyle\sum_{h \in \text{ablated}} (DE_h^{\text{clean}} - DE_h^{\text{abl}})}_{\text{direct}} - \underbrace{(LD^{\text{clean}} - LD^{\text{abl}})}_{\text{total}}$$

Note: DE uses each run's own final-LN scale, so "repair" includes LayerNorm rescaling as well as heads changing their outputs. Separating
the two (by freezing the LN scale) is a good extension.

In [ ]:
ablator = Ablator(model, ds, corrupt_cache)
NAME_MOVERS = [(9, 9), (9, 6), (10, 0)]

r = hydra(ablator, NAME_MOVERS, "mean", clean_ld, de_clean)
print("direct (naive prediction)", fmt_ci(bootstrap_ci(r["direct"])))
print("total  (actual drop)     ", fmt_ci(bootstrap_ci(r["total"])))
print("repair fraction          ", fmt_ci(bootstrap_ci(r["repair"], r["direct"], stat=lambda a, b: a.mean() / b.mean())))

heatmap(r["delta_de"].mean(0), "Change in direct effect after mean-ablating name movers", annotate_top=8)
print("\ntop compensators (ΔDE):")
for l, h, v in top_heads(r["delta_de"].mean(0), 8):
    print(f"  {l}.{h:<3} {fmt_ci(bootstrap_ci(r['delta_de'][:, l, h]))}")
savefig("hydra_delta_de")

In [ ]:
# Full accounting: does the change in head DEs explain the change in LD?
ablated_logits, ablated_cache = ablator.run(NAME_MOVERS, "mean", with_cache=True)
d = r["delta_de"].mean(0)
lost = sum(d[l, h] for l, h in NAME_MOVERS).item()
print(f"ablated heads {lost:+.2f} | other heads {d.sum().item() - lost:+.2f} | "
      f"remainder (MLPs, embeddings, biases, LN) {-r['total'].mean().item() - d.sum().item():+.2f} | "
      f"net change in LD {-r['total'].mean().item():+.2f}")
b = torch.arange(len(ds))
print(f"final LN scale: clean {clean_cache['ln_final.hook_scale'][b, ds.end].mean():.2f}, "
      f"ablated {ablated_cache['ln_final.hook_scale'][b, ds.end].mean():.2f}")

de_abl = de_clean + r["delta_de"]
print("\nhead   DE clean -> ablated   attn END->IO clean -> ablated")
for l, h in [(10, 7), (10, 10), (10, 2), (11, 2), (10, 6), (10, 1), (11, 10)]:
    print(f"{l}.{h:<4} {de_clean[:, l, h].mean():+.2f} -> {de_abl[:, l, h].mean():+.2f}        "
          f"{attention_to(clean_cache, ds, l, h, 'IO').mean():.2f} -> {attention_to(ablated_cache, ds, l, h, 'IO').mean():.2f}")

In [ ]:
# Waterfall: clean LD -> ablated LD, split into who changed their direct effect.
BACKUPS = [(10, 10), (10, 2), (11, 2), (10, 6), (10, 1)]
d = r["delta_de"].mean(0)
named = NAME_MOVERS + [(10, 7), (11, 10)] + BACKUPS
steps = [
    ("name movers\nablated", sum(d[hd] for hd in NAME_MOVERS).item()),
    ("brake 10.7\nreleases", d[10, 7].item()),
    ("backups\nstep up", sum(d[hd] for hd in BACKUPS).item()),
    ("brake 11.10\ntightens", d[11, 10].item()),
    ("other\nheads", d.sum().item() - sum(d[hd] for hd in named).item()),
    ("MLPs, biases,\nLayerNorm", -r["total"].mean().item() - d.sum().item()),
]
start, end = clean_ld.mean().item(), r["ld"].mean().item()
for name, v in steps:
    print(f"{name.replace(chr(10), ' '):24} {v:+.2f}")
print(f"clean LD {start:.2f} -> ablated LD {end:.2f}")

GAIN, LOSS, TOTAL = "#2a78d6", "#eb6834", "#8a8984"
fig, ax = plt.subplots(figsize=(10, 4.2))
labels = ["clean LD"] + [s[0] for s in steps] + ["ablated LD"]
ax.bar(0, start, 0.6, color=TOTAL, zorder=2)
level = start
for k, (name, v) in enumerate(steps, start=1):
    ax.bar(k, v, 0.6, bottom=level, color=GAIN if v >= 0 else LOSS, zorder=2)
    ax.plot([k - 1 + 0.3, k - 0.3], [level, level], color="#52514e", lw=0.8, zorder=1)
    ax.text(k, max(level, level + v) + 0.12, f"{v:+.2f}", ha="center", va="bottom", fontsize=9, color="#0b0b0b")
    level += v
ax.plot([len(steps) + 0.3 - 1, len(steps) + 1 - 0.3], [level, level], color="#52514e", lw=0.8, zorder=1)
ax.bar(len(steps) + 1, end, 0.6, color=TOTAL, zorder=2)
for k, v in [(0, start), (len(steps) + 1, end)]:
    ax.text(k, v + 0.12, f"{v:.2f}", ha="center", va="bottom", fontsize=9, color="#0b0b0b")
ax.set_xticks(range(len(labels)), labels, fontsize=9)
lo = min(0.0, start + steps[0][1])
ax.set(ylabel="logit difference", ylim=(lo - 0.4, max(start, end) + 0.9))
ax.set_title("Where the repair comes from: mean-ablating the name movers", pad=14)
ax.axhline(0, color="#52514e", lw=0.8)
ax.grid(axis="y", color="#e5e4df", zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
savefig("repair_waterfall")

**Things to interpret:**
- How much of the repair comes from the *negative* name mover 10.7 becoming less negative, and how much from backup name movers becoming more positive?
  (Compare McDougall et al. 2023, [Copy Suppression](https://arxiv.org/abs/2310.04625).)
- Summed over all heads, does ΔDE account for the repair, or is a big share in MLPs and LN scale?

## 6. Additivity

Two different ways the network can be non-additive:
- **direct ≠ total** (§5): the network adjusts *downstream* of the ablation.
- **Σ singles ≠ joint**: ablating heads one at a time does not add up to ablating them together, so the heads interact.

Interaction = Σ singles − joint, bootstrapped per prompt (paired).

In [ ]:
HEAD_SETS = {
    "name movers": NAME_MOVERS,
    "NM + negative NM": NAME_MOVERS + [(10, 7), (11, 10)],
    "NM + backup NM": NAME_MOVERS + [(10, 10), (10, 6), (10, 2), (10, 1), (11, 2), (9, 7), (9, 0), (11, 9)],
    "S-inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
}
rows = []
for name, heads in HEAD_SETS.items():
    for kind in ABLATION_KINDS:
        a = additivity(ablator, heads, kind, clean_ld)
        s, j = a["singles"].sum(-1), a["joint"]
        rows.append((name, kind, bootstrap_ci(s), bootstrap_ci(j), bootstrap_ci(s - j)))
        print(f"{name:18} {kind:9} Σsingles {fmt_ci(rows[-1][2]):22} joint {fmt_ci(rows[-1][3]):22} interaction {fmt_ci(rows[-1][4])}")

## 7. Systematics

**(a) Ablation method.** Zero, mean (per template and position, over ABC prompts), and resample (the matched ABC prompt) ask different
counterfactual questions. Zero ablation pushes activations off-distribution. Treat the spread as a systematic uncertainty on the headline number.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
w = 0.38
for i, kind in enumerate(ABLATION_KINDS):
    rk = hydra(ablator, NAME_MOVERS, kind, clean_ld, de_clean)
    print(f"{kind:9} direct {fmt_ci(bootstrap_ci(rk['direct']))}  total {fmt_ci(bootstrap_ci(rk['total']))}  "
          f"repair fraction {fmt_ci(bootstrap_ci(rk['repair'], rk['direct'], stat=lambda a, b: a.mean() / b.mean()))}")
    for off, key, color in [(-w / 2, "direct", "C0"), (w / 2, "total", "C3")]:
        p, lo, hi = bootstrap_ci(rk[key])
        ax.bar(i + off, p, w, yerr=[[p - lo], [hi - p]], color=color, capsize=3, label=key if i == 0 else None)
ax.axhline(0, color="k", lw=0.5)
ax.set(xticks=range(len(ABLATION_KINDS)), xticklabels=ABLATION_KINDS, ylabel="drop in LD",
       title="Ablating name movers: predicted (direct) vs actual (total)")
ax.legend();
savefig("direct_vs_total")

**(b) Template** and **(c) dataset seed**: does the repair fraction move by more than its statistical error?

In [ ]:
frac = lambda a, b: a.mean() / b.mean()
r = hydra(ablator, NAME_MOVERS, "mean", clean_ld, de_clean)
for t in range(len(TEMPLATES)):
    m = ds.template_id == t
    print(f"template {t}: repair fraction {fmt_ci(bootstrap_ci(r['repair'][m], r['direct'][m], stat=frac))}")

In [ ]:
for seed in range(1, 4):
    ds_s = make_ioi_dataset(model, n_per_template=16, seed=seed)
    cl_s, cc_s = model.run_with_cache(ds_s.clean_tokens)
    _, kc_s = model.run_with_cache(ds_s.corrupt_tokens)
    ld_s = logit_diff(cl_s, ds_s)
    r_s = hydra(Ablator(model, ds_s, kc_s), NAME_MOVERS, "mean", ld_s, head_direct_effects(model, cc_s, ds_s))
    print(f"seed {seed}: clean LD {ld_s.mean():.2f}  repair fraction {fmt_ci(bootstrap_ci(r_s['repair'], r_s['direct'], stat=frac))}")

## 8. Write-up (fill in)

- **Headline:** Mean-ablating GPT-2 small's three IOI name-mover heads removes ___ ± ___ of direct logit difference, but the actual
  logit difference drops by only ___ ± ___. The network repairs ___% of the damage.
- **Who repairs:** ___ (10.7, the negative name mover, contributes ___; backup name movers contribute ___).
- **Interactions:** Σ singles vs joint: ___
- **Systematics:** ablation method moves the repair fraction by ___, templates by ___, seeds by ___ (statistical error: ___).
- **So what:** a single-head ablation (or a direct effect alone) can badly misstate a component's importance. For Goodfire-style work
  (probes as rewards, Silico running autonomous ablation studies), effect sizes need both the counterfactual choice and the
  network's compensation reported alongside them.
- **Next:** freeze the LN scale to separate LN rescaling from head compensation; path patching to see *how* 10.7 detects the ablation; repeat on Pythia or Gemma-2-2B.